In [ ]:
import numpy as np
import pandas as pd


# Initial stage: calculating women's rating

In [4]:
df = pd.read_csv("../data/raw/WRegularSeasonCompactResults.csv")
df.head()

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT
0,1998,18,3104,91,3202,41,H,0
1,1998,18,3163,87,3221,76,H,0
2,1998,18,3222,66,3261,59,H,0
3,1998,18,3307,69,3365,62,H,0
4,1998,18,3349,115,3411,35,H,0


In [25]:
# 1. Get all unique women's teams and create the index mapping
women_team_id = pd.concat([df['WTeamID'], df['LTeamID']]).unique()
num_teams = len(women_team_id)
team_to_idx = {team_id: idx for idx, team_id in enumerate(women_team_id)}
for k, v in team_to_idx.items():
    print(f"{k}: {v}")

3104: 0
3163: 1
3222: 2
3307: 3
3349: 4
3435: 5
3443: 6
3119: 7
3139: 8
3141: 9
3142: 10
3182: 11
3184: 12
3207: 13
3279: 14
3339: 15
3353: 16
3408: 17
3427: 18
3436: 19
3448: 20
3453: 21
3454: 22
3124: 23
3133: 24
3151: 25
3170: 26
3199: 27
3231: 28
3234: 29
3242: 30
3269: 31
3274: 32
3301: 33
3304: 34
3321: 35
3326: 36
3345: 37
3347: 38
3361: 39
3385: 40
3420: 41
3438: 42
3452: 43
3464: 44
3156: 45
3178: 46
3248: 47
3261: 48
3281: 49
3283: 50
3320: 51
3368: 52
3422: 53
3106: 54
3117: 55
3120: 56
3187: 57
3203: 58
3210: 59
3235: 60
3250: 61
3266: 62
3323: 63
3344: 64
3356: 65
3380: 66
3396: 67
3397: 68
3407: 69
3419: 70
3459: 71
3462: 72
3463: 73
3137: 74
3164: 75
3181: 76
3241: 77
3259: 78
3284: 79
3298: 80
3314: 81
3336: 82
3376: 83
3400: 84
3103: 85
3130: 86
3143: 87
3157: 88
3177: 89
3193: 90
3196: 91
3197: 92
3211: 93
3217: 94
3322: 95
3389: 96
3409: 97
3110: 98
3122: 99
3168: 100
3179: 101
3186: 102
3191: 103
3208: 104
3218: 105
3220: 106
3243: 107
3247: 108
3251: 109
3270: 110


In [23]:
# 2. Initialize the empty matrix A and vector b
A = np.zeros((num_teams, num_teams))
b = np.zeros(num_teams)

In [26]:
# 3. Loop through the filtered historical dataframe (e.g., just the last 5 years)
for _, row in df.iterrows():
    w_team = row['WTeamID']
    l_team = row['LTeamID']
    margin = row['WScore'] - row['LScore']

    # get matrix indices 
    w_idx = team_to_idx[w_team]
    l_idx = team_to_idx[l_team]

    # applying logic 
    # diagnoal (games played)
    A[w_idx, w_idx] += 1
    A[l_idx, l_idx] += 1

    # off diagonal (opponents)
    A[w_idx, l_idx] -= 1
    A[l_idx, w_idx] -= 1

    # Vector b (Net margins)
    b[w_idx] += margin
    b[l_idx] -= margin


# 4. Solve for the true ratings using Least Squares (to avoid the singular matrix trap)
ratings, residuals, rank, s = np.linalg.lstsq(A, b, rcond=None)

# 5. Map the ratings back to the Team IDs so we can use them
team_ratings = {women_team_id[i]: ratings[i] for i in range(num_teams)}

In [28]:
for k, v in team_ratings.items():
    print(f"{k}: {v}")

3104: 12.550071674174832
3163: 41.51365539513128
3222: 3.834329664565928
3307: 10.226815150114387
3349: 5.923887947363827
3435: 19.402758016818453
3443: 7.927834917455087
3119: -4.376000944992818
3139: 1.877318407226805
3141: 1.3320056039727017
3142: -2.43522053363527
3182: 4.1916159085624125
3184: -5.718315711777569
3207: 8.212341439120095
3279: 12.581810121600208
3339: 1.5573357816674465
3353: 16.940113317120034
3408: 8.784297745293653
3427: -2.5092396321982324
3436: -2.8161001083892256
3448: 8.687126860227572
3453: 13.830408690141152
3454: -0.18352173060227805
3124: 30.555974126969883
3133: -1.8803320722088745
3151: 3.7234827354279703
3170: -8.911757024146791
3199: 19.473200596063904
3231: 15.482645184140361
3234: 19.925337658521087
3242: 13.21346506485753
3269: -0.20843186510611234
3274: 15.595882959308144
3301: 20.363173818276053
3304: 17.365735991281774
3321: 7.977205667471182
3326: 22.999715094608764
3345: 18.225001542446037
3347: -8.167240159554469
3361: 2.6949390433062863
3385

In [ ]:
# 1. load submission file path template
file_path = "../data/raw/SampleSubmissionStage2.csv"
sub = pd.read_csv(file_path)

# 2. The Scaling Factor (k)
# We use 0.15 to flatten the curve and protect our Brier Score from massive upset penalties.
k = 0.15

def get_win_probability(rating_a, rating_b):
    """Calculates the probability of Team A beating Team B using a scaled sigmoid."""
    diff = rating_a - rating_b
    return 1 / (1 / np.exp(-k * diff))

predictions = []

# 3. pipeline loop
for inde, row in sub.iterrows():
    # Parse the Kaggle ID string (e.g., "2026_1101_1102")
    id_parts = row['ID'].split('_')
    team_a = int(id_parts[1]) # lower ID
    team_b = int(id_parts[2]) # Higher ID

    # 4. routing logic 
    if team_a < 2000:
        # mean's tour (ID 1000 -1999)
        # Placeholder: We output 0.50 to avoid crashing, minimizing Brier Score damage.
        pred = 0.5
    else:
        # Women's tour (IDs 3000-3999)
        # Use .get(team_id, 0) as a safety net in case a team has zero historical games
        rating_a = team_ratings.get(team_a, 0)
        rating_b = team_ratings.get(team_b, 0)
        Pred = get_win_probability(rating_a, rating_b)

    predictions.append(pred)

# 5. compile and save
sub['pred'] = predictions
sub.to_csv('submission.csv', index=False)
print("Pipeline complete. submission.csv generated successfully.")

Pipeline complete. submission.csv generated successfully.


In [32]:
sub.head()

,ID,Pred,pred
0,2026_1101_1102,0.5,0.5
1,2026_1101_1103,0.5,0.5
2,2026_1101_1104,0.5,0.5
3,2026_1101_1105,0.5,0.5
4,2026_1101_1106,0.5,0.5


In [34]:
if 'pred' in sub.columns:
    sub['Pred'] = sub['pred']

# 2. Save ONLY the 'ID' and 'Pred' columns to the final CSV
sub[['ID', 'Pred']].to_csv('submission.csv', index=False)

# 3. Verify the final shape (Should print just two columns)
print(sub[['ID', 'Pred']].tail())

                    ID      Pred
132128  2026_3478_3480  4.618930
132129  2026_3478_3481  1.366324
132130  2026_3479_3480  4.762364
132131  2026_3479_3481  1.408754
132132  2026_3480_3481  0.295810


In [ ]:
pred = []
for index, row in sub.iterrows():
    # ID looks like "2026_1101_1102"
    id_parts = row['ID'].split('_')
    team_a = int(id_parts[1])
    team_b = int(id_parts[2])

    # routing logic
    if team_a < 2000:
        # belongs to men
        pred = model_m.predict_proba(get_mens_features(team_a, team_b))
    else:
        # belongs to women
        pred = model_w.predict_proba(get_womens_features(team_a, team_b))
    
    # combine result 
    predictions.append()

sub['Pred'] = predictions
sub.to_csv('submission.csv', index=False)

# stage 2:

In [5]:
import sys
sys.path.append('../src/')

from data_loader import load_data
import pandas as np

In [14]:
m_reg_comp = load_data('../data/raw/MRegularSeasonCompactResults.csv')
m_reg_detailed = load_data('../data/raw/MRegularSeasonDetailedResults.csv')
w_reg_comp = load_data('../data/raw/WRegularSeasonCompactResults.csv')
w_reg_detailed = load_data('../data/raw/WRegularSeasonDetailedResults.csv')

In [15]:
# men regular season detailed
m_reg_detailed.columns

Index(['Season', 'DayNum', 'WTeamID', 'WScore', 'LTeamID', 'LScore', 'WLoc',
       'NumOT', 'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR',
       'WAst', 'WTO', 'WStl', 'WBlk', 'WPF', 'LFGM', 'LFGA', 'LFGM3', 'LFGA3',
       'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF'],
      dtype='object')

In [27]:
m_reg_detailed['WPos'] = m_reg_detailed['WFGA'] - m_reg_detailed['WOR'] + m_reg_detailed['WTO'] + (0.44 * m_reg_detailed['WFTA'])
m_reg_detailed['LPos'] = m_reg_detailed['LFGA'] - m_reg_detailed['LOR'] + m_reg_detailed['LTO'] + (0.44 * m_reg_detailed['LFTA'])
m_reg_detailed['GamePos'] =(m_reg_detailed['WPos'] + m_reg_detailed['LPos']) / 2
m_reg_detailed[['WPos', 'LPos', 'GamePos']]

,WPos,LPos,GamePos
0,74.92,70.68,72.80
1,68.36,67.80,68.08
2,63.76,64.12,63.94
3,57.64,57.60,57.62
4,63.72,62.88,63.30
...,...,...,...
122770,72.24,73.48,72.86
122771,68.56,67.36,67.96
122772,62.64,62.84,62.74
122773,65.28,60.60,62.94


In [26]:
m_reg_detailed[['WScore', 'LScore']]

,WScore,LScore
0,68,62
1,70,63
2,73,61
3,56,50
4,77,71
...,...,...
122770,80,78
122771,81,67
122772,90,61
122773,77,62


In [28]:
# Winning Team Efficiency
m_reg_detailed['WORtg'] = (m_reg_detailed['WScore'] / m_reg_detailed['GamePos']) * 100
m_reg_detailed['WDRtg'] = (m_reg_detailed['LScore'] / m_reg_detailed['GamePos']) * 100

# Losing Team Efficiency (Notice how the scores flip for the defense)
m_reg_detailed['LORtg'] = (m_reg_detailed['LScore'] / m_reg_detailed['GamePos']) * 100
m_reg_detailed['LDRtg'] = (m_reg_detailed['WScore'] / m_reg_detailed['GamePos']) * 100

In [39]:
# Turn game centric into team centric
# 1 & 3: Isolate Winners and rename columns
winners = m_reg_detailed[['Season', 'WTeamID', 'WORtg', 'WDRtg']].copy()
winners.columns = ['Season', 'TeamID', 'ORtg', 'DRtg']
print(winners.head())
print()

# 2 & 3: Isolate Losers and rename columns
losers = m_reg_detailed[['Season', 'LTeamID', 'LORtg', 'LDRtg']].copy()
losers.columns = ['Season', 'TeamID', 'ORtg', 'DRtg']
print(len(losers) + len(winners))


   Season  TeamID        ORtg        DRtg
0    2003    1104   93.406593   85.164835
1    2003    1272  102.820212   92.538190
2    2003    1266  114.169534   95.401939
3    2003    1296   97.188476   86.775425
4    2003    1400  121.642970  112.164297

245550


In [36]:
import pandas as pd
all_team_games = pd.concat([winners, losers], ignore_index=True)
all_team_games

,Season,TeamID,ORtg,DRtg
0,2003,1104,93.406593,85.164835
1,2003,1272,102.820212,92.538190
2,2003,1266,114.169534,95.401939
3,2003,1296,97.188476,86.775425
4,2003,1400,121.642970,112.164297
...,...,...,...,...
245545,2026,1347,107.054625,109.799616
245546,2026,1440,98.587404,119.187758
245547,2026,1236,97.226650,143.449155
245548,2026,1355,98.506514,122.338735


In [98]:
team_season_avg = all_team_games.groupby(['Season', 'TeamID'])[['ORtg', 'DRtg']].mean().reset_index()
team_season_avg['NetRating'] = team_season_avg['ORtg'] - team_season_avg['DRtg']
team_season_avg

,Season,TeamID,ORtg,DRtg,NetRating
0,2003,1102,105.037129,104.828579,0.208550
1,2003,1103,111.980696,111.703480,0.277217
2,2003,1104,104.603582,98.796544,5.807039
3,2003,1105,94.623503,101.263945,-6.640442
4,2003,1106,94.724360,95.172172,-0.447812
...,...,...,...,...,...
8341,2026,1477,97.664088,110.344924,-12.680836
8342,2026,1478,105.053725,108.074685,-3.020960
8343,2026,1479,100.264632,103.827388,-3.562757
8344,2026,1480,103.208399,114.996002,-11.787602


In [93]:
# 1. Isolate Winners (Now grabbing LTeamID as the Opponent)
winners = m_reg_detailed[['Season', 'WTeamID', 'LTeamID', 'WORtg', 'WDRtg']].copy()
winners.columns = ['Season', 'TeamID', 'OppID', 'ORtg', 'DRtg']

# 2. Isolate Losers (Now grabbing WTeamID as the Opponent)
losers = m_reg_detailed[['Season', 'LTeamID', 'WTeamID', 'LORtg', 'LDRtg']].copy()
losers.columns = ['Season', 'TeamID', 'OppID', 'ORtg', 'DRtg']

# 3. Stack them
all_team_games = pd.concat([winners, losers], ignore_index=True)
all_team_games

,Season,TeamID,OppID,ORtg,DRtg
0,2003,1104,1328,93.406593,85.164835
1,2003,1272,1393,102.820212,92.538190
2,2003,1266,1437,114.169534,95.401939
3,2003,1296,1457,97.188476,86.775425
4,2003,1400,1208,121.642970,112.164297
...,...,...,...,...,...
245545,2026,1347,1457,107.054625,109.799616
245546,2026,1440,1459,98.587404,119.187758
245547,2026,1236,1464,97.226650,143.449155
245548,2026,1355,1472,98.506514,122.338735


In [94]:
games_with_sos = pd.merge(
    all_team_games, 
    team_season_avg[['Season', 'TeamID', 'NetRating']], 
    left_on=['Season', 'OppID'],   # Match the Opponent in the game...
    right_on=['Season', 'TeamID'], # ...to the Team in the averages list
    how='left'                     # Keep all our game rows intact
)

# Rename the newly merged column so we don't get confused
games_with_sos.rename(columns={'NetRating': 'Opp_NetRating'}, inplace=True)

# Drop the extra TeamID column that came over from the right dataframe
games_with_sos.drop(columns=['TeamID_y'], errors='ignore', inplace=True)
games_with_sos.rename(columns={'TeamID_x': 'TeamID'}, errors='ignore', inplace=True)
games_with_sos

,Season,TeamID,OppID,ORtg,DRtg,Opp_NetRating
0,2003,1104,1328,93.406593,85.164835,16.436261
1,2003,1272,1393,102.820212,92.538190,13.833350
2,2003,1266,1437,114.169534,95.401939,2.678812
3,2003,1296,1457,97.188476,86.775425,3.973739
4,2003,1400,1208,121.642970,112.164297,8.190653
...,...,...,...,...,...,...
245545,2026,1347,1457,107.054625,109.799616,6.123504
245546,2026,1440,1459,98.587404,119.187758,0.998231
245547,2026,1236,1464,97.226650,143.449155,2.603592
245548,2026,1355,1472,98.506514,122.338735,9.959952


In [95]:
sos_df = games_with_sos.groupby(['Season', 'TeamID'])['Opp_NetRating'].mean().reset_index()
sos_df

,Season,TeamID,Opp_NetRating
0,2003,1102,1.123343
1,2003,1103,-1.494825
2,2003,1104,5.176372
3,2003,1105,-5.842197
4,2003,1106,-3.193068
...,...,...,...
8341,2026,1477,-0.029096
8342,2026,1478,-5.765916
8343,2026,1479,-6.646097
8344,2026,1480,-2.958063


In [119]:
team_season_avg = team_season_avg.merge(sos_df)
team_season_avg['AdjNetRating'] = team_season_avg['NetRating'] + team_season_avg['Opp_NetRating']
team_season_avg

,Season,TeamID,ORtg,DRtg,NetRating,Opp_NetRating,AdjNetRating,MedianRank_x,MedianRank_y
0,2003,1102,105.037129,104.828579,0.208550,1.123343,1.331893,155.5,155.5
1,2003,1103,111.980696,111.703480,0.277217,-1.494825,-1.217608,170.5,170.5
2,2003,1104,104.603582,98.796544,5.807039,5.176372,10.983411,37.0,37.0
3,2003,1105,94.623503,101.263945,-6.640442,-5.842197,-12.482639,310.0,310.0
4,2003,1106,94.724360,95.172172,-0.447812,-3.193068,-3.640880,265.5,265.5
...,...,...,...,...,...,...,...,...,...
8340,2026,1477,97.664088,110.344924,-12.680836,-0.029096,-12.709931,294.0,294.0
8341,2026,1478,105.053725,108.074685,-3.020960,-5.765916,-8.786876,287.0,287.0
8342,2026,1479,100.264632,103.827388,-3.562757,-6.646097,-10.208854,291.0,291.0
8343,2026,1480,103.208399,114.996002,-11.787602,-2.958063,-14.745665,307.0,307.0


In [100]:
# Get unique teams for a specific season
teams = sorted(all_team_games[all_team_games['Season'] == 2024]['TeamID'].unique())
team_map = {team_id: i for i, team_id in enumerate(teams)}
num_teams = len(teams)

In [103]:
import numpy as np

M = np.zeros((num_teams, num_teams))
y = np.zeros(num_teams)

# Fill the matrix based on game results
# We'll use the 'm_reg_detailed' for a specific season
season_data = m_reg_detailed[m_reg_detailed['Season'] == 2024]

for _, row in season_data.iterrows():
    w_idx = team_map[row['WTeamID']]
    l_idx = team_map[row['LTeamID']]
    # margin = row['WScore'] - row['LScore']
    margin = row['WORtg'] - row['WDRtg'] # improved
    
    # Update Diagonal (Total games played)
    M[w_idx, w_idx] += 1
    M[l_idx, l_idx] += 1
    
    # Update Off-Diagonal (Games played against each other)
    M[w_idx, l_idx] -= 1
    M[l_idx, w_idx] -= 1
    
    # Update Point Differential Vector
    y[w_idx] += margin
    y[l_idx] -= margin

# THE MASSEY FIX: Replace last row to make it solvable
M[-1, :] = 1
y[-1] = 0

In [104]:
ratings = np.linalg.solve(M, y)
massey_results = pd.DataFrame({'TeamID': teams, 'Massey_Adj_Eff': ratings})

In [106]:
massey_results

,TeamID,Massey_Adj_Eff
0,1101,-4.851367
1,1102,-7.281489
2,1103,5.017018
3,1104,30.221708
4,1105,-19.449695
...,...,...
357,1474,-10.872087
358,1475,-18.451285
359,1476,-27.350334
360,1477,-19.908954


# massey

In [109]:
m_massey = load_data('../data/raw/MMasseyOrdinals.csv')

In [110]:
# 1. Find the maximum day each system reported for each season
last_day_per_system = m_massey.groupby(['Season', 'SystemName'])['RankingDayNum'].max().reset_index()

# 2. Merge this back to the original data to filter for only those 'last days'
final_ranks_raw = pd.merge(
    m_massey, 
    last_day_per_system, 
    on=['Season', 'SystemName', 'RankingDayNum'], 
    how='inner'
)

# 3. Now, "The Wisdom of Crowds": Calculate the Median Rank per Team-Season
final_expert_ranks = final_ranks_raw.groupby(['Season', 'TeamID'])['OrdinalRank'].median().reset_index()

# 4. Rename for clarity
final_expert_ranks.columns = ['Season', 'TeamID', 'MedianRank']

In [111]:
final_expert_ranks

,Season,TeamID,MedianRank
0,2003,1102,155.5
1,2003,1103,170.5
2,2003,1104,37.0
3,2003,1105,310.0
4,2003,1106,265.5
...,...,...,...
8340,2026,1477,294.0
8341,2026,1478,287.0
8342,2026,1479,291.0
8343,2026,1480,307.0


In [126]:
# 1. Fill the holes in your 2026 'Brain'
# If an expert didn't rank them, they are effectively rank #350
team_season_avg['MedianRank'] = team_season_avg['MedianRank'].fillna(350)

# 2. (Optional but Safe) If any teams are missing AdjNet (e.g., they didn't play games)
# we set them to 0 (Average)
team_season_avg['AdjNetRating'] = team_season_avg['AdjNetRating'].fillna(0)

# 3. NOW perform the merges we discussed
# (Re-run the sub.merge steps here...)

In [127]:
print(sub[['Delta_AdjNet', 'Delta_Rank']].isnull().sum())

Delta_AdjNet    65703
Delta_Rank      65703
dtype: int64


In [129]:
print("Submission IDs (First 5):", sub['T1'].head().values)
print("Brain IDs (First 5):", team_season_avg['TeamID'].head().values)

Submission IDs (First 5): [1101 1101 1101 1101 1101]
Brain IDs (First 5): [1102 1103 1104 1105 1106]


In [130]:
# 1. How many teams does the submission want for 2026?
sub_teams = set(sub['T1'].unique())
print(f"Teams in Submission: {len(sub_teams)}")

# 2. How many teams does your Brain have for 2026?
brain_2026 = team_season_avg[team_season_avg['Season'] == 2026]
brain_teams = set(brain_2026['TeamID'].unique())
print(f"Teams in Brain (2026): {len(brain_teams)}")

# 3. What is the intersection?
overlap = sub_teams.intersection(brain_teams)
print(f"Teams that exist in BOTH: {len(overlap)}")

Teams in Submission: 726
Teams in Brain (2026): 365
Teams that exist in BOTH: 364


In [113]:
# 1. Start with the core game info
matchups = m_reg_detailed[['Season', 'WTeamID', 'LTeamID']].copy()

# 2. Merge the Winner's Season Stats
matchups = matchups.merge(
    team_season_avg[['Season', 'TeamID', 'AdjNetRating', 'MedianRank']],
    left_on=['Season', 'WTeamID'],
    right_on=['Season', 'TeamID'],
    how='left'
).rename(columns={'AdjNetRating': 'W_AdjNet', 'MedianRank': 'W_Rank'}).drop(columns='TeamID')

# 3. Merge the Loser's Season Stats
matchups = matchups.merge(
    team_season_avg[['Season', 'TeamID', 'AdjNetRating', 'MedianRank']],
    left_on=['Season', 'LTeamID'],
    right_on=['Season', 'TeamID'],
    how='left'
).rename(columns={'AdjNetRating': 'L_AdjNet', 'MedianRank': 'L_Rank'}).drop(columns='TeamID')

In [114]:
matchups

,Season,WTeamID,LTeamID,W_AdjNet,W_Rank,L_AdjNet,L_Rank
0,2003,1104,1328,10.983411,37.0,21.696054,6.0
1,2003,1272,1393,13.772646,21.0,18.185100,12.5
2,2003,1266,1437,19.341553,12.0,7.500256,73.0
3,2003,1296,1457,0.413652,118.0,-2.012765,191.0
4,2003,1400,1208,19.925362,6.0,15.234092,12.0
...,...,...,...,...,...,...,...
122770,2026,1457,1347,4.676484,126.0,-5.106512,243.0
122771,2026,1459,1440,-3.707453,194.0,-23.997951,355.0
122772,2026,1464,1236,-1.664720,233.0,-8.170941,185.0
122773,2026,1472,1355,5.663858,130.0,-1.563306,203.0


In [115]:
# Create the "Winner was Team 1" half
df_win = pd.DataFrame()
df_win['Delta_AdjNet'] = matchups['W_AdjNet'] - matchups['L_AdjNet']
df_win['Delta_Rank'] = matchups['W_Rank'] - matchups['L_Rank']
df_win['Result'] = 1

# Create the "Winner was Team 2" half (The Mirror)
df_loss = pd.DataFrame()
df_loss['Delta_AdjNet'] = matchups['L_AdjNet'] - matchups['W_AdjNet']
df_loss['Delta_Rank'] = matchups['L_Rank'] - matchups['W_Rank']
df_loss['Result'] = 0

# Combine them into the master Training Set
train_df = pd.concat([df_win, df_loss], axis=0).reset_index(drop=True)

In [121]:
# Drop any rows where we still have NaNs after the merge
train_df = train_df.dropna()

print(f"Rows remaining after cleaning: {len(train_df)}")

Rows remaining after cleaning: 245530


In [122]:
train_df.isnull().sum()

Delta_AdjNet    0
Delta_Rank      0
Result          0
dtype: int64

# Training

In [123]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, log_loss

# 1. Define Features and Target
X = train_df[['Delta_AdjNet', 'Delta_Rank']]
y = train_df['Result']

# 2. Initialize and Train
# We set fit_intercept=False because our data is perfectly symmetric 
# around zero; the baseline probability for two equal teams should be exactly 0.5
model = LogisticRegression(fit_intercept=False)
model.fit(X, y)

# 3. Extract the "Knowledge"
beta_adjnet = model.coef_[0][0]
beta_rank = model.coef_[0][1]

print(f"Coefficient for Delta_AdjNet: {beta_adjnet:.4f}")
print(f"Coefficient for Delta_Rank: {beta_rank:.4f}")

Coefficient for Delta_AdjNet: 0.0244
Coefficient for Delta_Rank: -0.0131


In [143]:
# 1. Fill missing stats with 'Neutral' values
# This ensures that Women vs Women matchups don't result in NaNs
sub['Delta_AdjNet'] = sub['Delta_AdjNet'].fillna(0)
sub['Delta_Rank'] = sub['Delta_Rank'].fillna(0)

# 2. Verify the NaNs are gone
print(f"Remaining NaNs in Deltas: {sub['Delta_AdjNet'].isnull().sum()}")

# 3. Predict!
X_test = sub[['Delta_AdjNet', 'Delta_Rank']]
sub['Pred'] = model.predict_proba(X_test)[:, 1]

# 4. Apply the 'Clipping' we discussed to be safe
sub['Pred'] = sub['Pred'].clip(0.022, 0.978)

Remaining NaNs in Deltas: 0


In [144]:
sub.head()

,ID,Pred,Season,T1,T2,T1_AdjNet,T1_Rank,T2_AdjNet,T2_Rank,Delta_AdjNet,Delta_Rank
0,2026_1101_1102,0.808968,2026,1101,1102,-10.515809,251.0,-20.791279,342.0,10.275470,-91.0
1,2026_1101_1103,0.039317,2026,1101,1103,-10.515809,251.0,16.809219,58.0,-27.325028,193.0
2,2026_1101_1104,0.022000,2026,1101,1104,-10.515809,251.0,22.289301,20.0,-32.805110,231.0
3,2026_1101_1105,0.651031,2026,1101,1105,-10.515809,251.0,-14.587455,291.0,4.071646,-40.0
4,2026_1101_1106,0.757139,2026,1101,1106,-10.515809,251.0,-19.519726,321.0,9.003916,-70.0


In [145]:
sub[['ID', 'Pred']].to_csv('submission_2026_v1.csv', index=False)

In [146]:
pd.read_csv('submission_2026_v1.csv')

,ID,Pred
0,2026_1101_1102,0.808968
1,2026_1101_1103,0.039317
2,2026_1101_1104,0.022000
3,2026_1101_1105,0.651031
4,2026_1101_1106,0.757139
...,...,...
132128,2026_3478_3480,0.500000
132129,2026_3478_3481,0.500000
132130,2026_3479_3480,0.500000
132131,2026_3479_3481,0.500000


In [147]:
def build_efficiency_brain(results_df, ordinals_df):
    # 1. Basic Stats & Efficiency
    results_df['WPos'] = results_df['WFGA'] - results_df['WOR'] + results_df['WTO'] + (0.475 * results_df['WFTA'])
    results_df['LPos'] = results_df['LFGA'] - results_df['LOR'] + results_df['LTO'] + (0.475 * results_df['LFTA'])
    results_df['WORtg'] = (results_df['WScore'] / results_df['WPos']) * 100
    results_df['WDRtg'] = (results_df['LScore'] / results_df['WPos']) * 100
    results_df['LORtg'] = (results_df['LScore'] / results_df['LPos']) * 100
    results_df['LDRtg'] = (results_df['WScore'] / results_df['LPos']) * 100

    # 2. Transform to Team-Centric (Long Format)
    winners = results_df[['Season', 'WTeamID', 'LTeamID', 'WORtg', 'WDRtg']].copy()
    winners.columns = ['Season', 'TeamID', 'OppID', 'ORtg', 'DRtg']
    losers = results_df[['Season', 'LTeamID', 'WTeamID', 'LORtg', 'LDRtg']].copy()
    losers.columns = ['Season', 'TeamID', 'OppID', 'ORtg', 'DRtg']
    all_games = pd.concat([winners, losers], ignore_index=True)

    # 3. Raw Net Rating
    all_games['NetRating'] = all_games['ORtg'] - all_games['DRtg']
    team_season_avg = all_games.groupby(['Season', 'TeamID'])[['ORtg', 'DRtg', 'NetRating']].mean().reset_index()

    # 4. Strength of Schedule (SOS) Adjustment
    all_games = all_games.merge(team_season_avg[['Season', 'TeamID', 'NetRating']], 
                                left_on=['Season', 'OppID'], right_on=['Season', 'TeamID'], 
                                how='left', suffixes=('', '_opp'))
    sos_df = all_games.groupby(['Season', 'TeamID'])['NetRating_opp'].mean().reset_index()
    sos_df.rename(columns={'NetRating_opp': 'Opp_NetRating'}, inplace=True)
    
    # 5. Final Adjustment
    team_season_avg = team_season_avg.merge(sos_df, on=['Season', 'TeamID'])
    team_season_avg['AdjNetRating'] = team_season_avg['NetRating'] + team_season_avg['Opp_NetRating']

    # 6. Expert Brain (Median Rank)
    # Get max day rankings
    last_day = ordinals_df.groupby(['Season', 'SystemName'])['RankingDayNum'].max().reset_index()
    final_raw = ordinals_df.merge(last_day, on=['Season', 'SystemName', 'RankingDayNum'])
    expert_ranks = final_raw.groupby(['Season', 'TeamID'])['OrdinalRank'].median().reset_index()
    expert_ranks.columns = ['Season', 'TeamID', 'MedianRank']

    # 7. Merge and Impute
    final_brain = team_season_avg.merge(expert_ranks, on=['Season', 'TeamID'], how='left')
    final_brain['MedianRank'] = final_brain['MedianRank'].fillna(350)
    
    return final_brain

In [149]:
w_reg_detailed = load_data('../data/raw/WRegularSeasonDetailedResults.csv')
w_seeds = load_data('../data/raw/WNCAATourneySeeds.csv')

In [150]:
# 1. Calculate Possessions and Ratings for Women
w_reg_detailed['WPos'] = w_reg_detailed['WFGA'] - w_reg_detailed['WOR'] + w_reg_detailed['WTO'] + (0.475 * w_reg_detailed['WFTA'])
w_reg_detailed['LPos'] = w_reg_detailed['LFGA'] - w_reg_detailed['LOR'] + w_reg_detailed['LTO'] + (0.475 * w_reg_detailed['LFTA'])

w_reg_detailed['WORtg'] = (w_reg_detailed['WScore'] / w_reg_detailed['WPos']) * 100
w_reg_detailed['WDRtg'] = (w_reg_detailed['LScore'] / w_reg_detailed['WPos']) * 100
w_reg_detailed['LORtg'] = (w_reg_detailed['LScore'] / w_reg_detailed['LPos']) * 100
w_reg_detailed['LDRtg'] = (w_reg_detailed['WScore'] / w_reg_detailed['LPos']) * 100

# 2. Transform to Team-Centric format
w_winners = w_reg_detailed[['Season', 'WTeamID', 'LTeamID', 'WORtg', 'WDRtg']].copy()
w_winners.columns = ['Season', 'TeamID', 'OppID', 'ORtg', 'DRtg']
w_losers = w_reg_detailed[['Season', 'LTeamID', 'WTeamID', 'LORtg', 'LDRtg']].copy()
w_losers.columns = ['Season', 'TeamID', 'OppID', 'ORtg', 'DRtg']
w_all_games = pd.concat([w_winners, w_losers], ignore_index=True)

# 3. Basic NetRating
w_all_games['NetRating'] = w_all_games['ORtg'] - w_all_games['DRtg']
w_team_season_avg = w_all_games.groupby(['Season', 'TeamID'])[['ORtg', 'DRtg', 'NetRating']].mean().reset_index()

# 4. Strength of Schedule (SOS) Adjustment
w_all_games = w_all_games.merge(w_team_season_avg[['Season', 'TeamID', 'NetRating']], 
                                left_on=['Season', 'OppID'], right_on=['Season', 'TeamID'], 
                                how='left', suffixes=('', '_opp'))
w_sos_df = w_all_games.groupby(['Season', 'TeamID'])['NetRating_opp'].mean().reset_index()
w_sos_df.rename(columns={'NetRating_opp': 'Opp_NetRating'}, inplace=True)

w_team_season_avg = w_team_season_avg.merge(w_sos_df, on=['Season', 'TeamID'])
w_team_season_avg['AdjNetRating'] = w_team_season_avg['NetRating'] + w_team_season_avg['Opp_NetRating']

In [152]:
# 2. Extract numeric seed (W01 -> 1) and scale it
w_seeds['SeedNum'] = w_seeds['Seed'].apply(lambda x: int(''.join(filter(str.isdigit, x))))
w_seeds['MedianRank'] = w_seeds['SeedNum'] * 4 # Scaling to match Massey range

# 3. Merge into the Women's Brain and Impute the rest
women_brain = w_team_season_avg.merge(w_seeds[['Season', 'TeamID', 'MedianRank']], on=['Season', 'TeamID'], how='left')
women_brain['MedianRank'] = women_brain['MedianRank'].fillna(150) # Penalty for unranked teams

In [153]:
women_brain

,Season,TeamID,ORtg,DRtg,NetRating,Opp_NetRating,AdjNetRating,MedianRank
0,2010,3102,80.215834,111.484300,-31.268466,3.113317,-28.155149,150.0
1,2010,3103,90.895518,87.092504,3.803015,-0.601821,3.201194,150.0
2,2010,3104,85.629519,89.693350,-4.063831,2.472300,-1.591532,150.0
3,2010,3105,82.391703,88.073814,-5.682111,-5.168971,-10.851082,150.0
4,2010,3106,79.464596,82.308995,-2.844399,-4.858214,-7.702613,150.0
...,...,...,...,...,...,...,...,...
5960,2026,3477,85.179951,92.875343,-7.695391,-5.326771,-13.022163,150.0
5961,2026,3478,80.233598,110.103365,-29.869767,-4.351310,-34.221077,150.0
5962,2026,3479,86.812513,108.550615,-21.738101,-2.284246,-24.022348,150.0
5963,2026,3480,94.525137,90.841672,3.683465,-8.586769,-4.903304,150.0


In [154]:
# This creates the 'men_brain' we were missing
men_brain = build_efficiency_brain(m_reg_detailed, m_massey)

In [155]:
men_brain

,Season,TeamID,ORtg,DRtg,NetRating,Opp_NetRating,AdjNetRating,MedianRank
0,2003,1102,103.754717,103.430091,0.324626,1.094412,1.419038,155.5
1,2003,1103,110.565133,110.240732,0.324401,-1.460746,-1.136345,170.5
2,2003,1104,103.369974,97.565124,5.804850,5.133689,10.938539,37.0
3,2003,1105,92.999174,99.549875,-6.550701,-5.851676,-12.402377,310.0
4,2003,1106,93.648220,94.062221,-0.414001,-3.196065,-3.610066,265.5
...,...,...,...,...,...,...,...,...
8341,2026,1477,96.614094,109.224951,-12.610857,-0.026080,-12.636937,294.0
8342,2026,1478,104.014805,107.024751,-3.009946,-5.660408,-8.670354,287.0
8343,2026,1479,100.253121,103.716712,-3.463592,-6.553798,-10.017390,291.0
8344,2026,1480,102.474067,114.065448,-11.591380,-2.883278,-14.474659,307.0


In [156]:
# Combine them into one "Universal Brain"
master_brain = pd.concat([men_brain, women_brain], ignore_index=True)

In [157]:
# Final Check: You should see 1000-series IDs AND 3000-series IDs
print(f"Men's teams: {len(master_brain[master_brain['TeamID'] < 2000])}")
print(f"Women's teams: {len(master_brain[master_brain['TeamID'] > 3000])}")

Men's teams: 8346
Women's teams: 5965


In [159]:
# 1. Start fresh with the submission file
sub = load_data('../data/raw/SampleSubmissionStage2.csv')
sub['Season'] = sub['ID'].apply(lambda x: int(x.split('_')[0]))
sub['T1'] = sub['ID'].apply(lambda x: int(x.split('_')[1]))
sub['T2'] = sub['ID'].apply(lambda x: int(x.split('_')[2]))

# 2. Merge the Master Brain for Team 1
sub = sub.merge(
    master_brain[['Season', 'TeamID', 'AdjNetRating', 'MedianRank']],
    left_on=['Season', 'T1'],
    right_on=['Season', 'TeamID'],
    how='left'
).rename(columns={'AdjNetRating': 'T1_AdjNet', 'MedianRank': 'T1_Rank'}).drop(columns='TeamID')

# 3. Merge the Master Brain for Team 2
sub = sub.merge(
    master_brain[['Season', 'TeamID', 'AdjNetRating', 'MedianRank']],
    left_on=['Season', 'T2'],
    right_on=['Season', 'TeamID'],
    how='left'
).rename(columns={'AdjNetRating': 'T2_AdjNet', 'MedianRank': 'T2_Rank'}).drop(columns='TeamID')

# 4. The Final "Safety Net" 
# If a team is still missing (very rare), we give them neutral stats
sub['T1_AdjNet'] = sub['T1_AdjNet'].fillna(0)
sub['T2_AdjNet'] = sub['T2_AdjNet'].fillna(0)
sub['T1_Rank'] = sub['T1_Rank'].fillna(150) 
sub['T2_Rank'] = sub['T2_Rank'].fillna(150)

# 5. Calculate the Deltas the model expects
sub['Delta_AdjNet'] = sub['T1_AdjNet'] - sub['T2_AdjNet']
sub['Delta_Rank'] = sub['T1_Rank'] - sub['T2_Rank']

# 6. Predict!
sub['Pred'] = model.predict_proba(sub[['Delta_AdjNet', 'Delta_Rank']])[:, 1]

# 7. Final Clip for Log-Loss safety (Don't skip this!)
sub['Pred'] = sub['Pred'].clip(0.022, 0.978)

# 8. Export Version 2
sub[['ID', 'Pred']].to_csv('submission_2026_v2_universal.csv', index=False)

In [161]:
sub.tail()

,ID,Pred,Season,T1,T2,T1_AdjNet,T1_Rank,T2_AdjNet,T2_Rank,Delta_AdjNet,Delta_Rank
132128,2026_3478_3480,0.328534,2026,3478,3480,-34.221077,150.0,-4.903304,150.0,-29.317773,0.0
132129,2026_3478_3481,0.419536,2026,3478,3481,-34.221077,150.0,-20.904780,150.0,-13.316297,0.0
132130,2026_3479_3480,0.385526,2026,3479,3480,-24.022348,150.0,-4.903304,150.0,-19.119044,0.0
132131,2026_3479_3481,0.481006,2026,3479,3481,-24.022348,150.0,-20.904780,150.0,-3.117568,0.0
132132,2026_3480_3481,0.596318,2026,3480,3481,-4.903304,150.0,-20.904780,150.0,16.001476,0.0


In [162]:
detailed_reg_df = load_data('../data/raw/MRegularSeasonDetailedResults.csv')

In [ ]:
# ==========================================
# 1. BUILD THE HISTORICAL BRAIN (2022 - 2024)
# ==========================================
# Strictly isolate the regular season data BEFORE the test year
train_years = [2022, 2023, 2024]
train_reg_df = detailed_reg_df[detailed_reg_df['Season'].isin(train_years)].copy()
train_ord_df = ordinals_df[ordinals_df['Season'].isin(train_years)].copy()

# Generate the Efficiency features
historical_brain = build_efficiency_brain(train_reg_df, train_ord_df)

# Prepare the Training Set by merging tournament outcomes with the Brain
train_tourney = tourney_df[tourney_df['Season'].isin(train_years)]

X_train_uni = []
y_train_uni = []

for _, row in train_tourney.iterrows():
    season = row['Season']
    w_team = row['WTeamID']
    l_team = row['LTeamID']
    
    # Extract features for Winner and Loser
    w_stats = historical_brain[(historical_brain['Season'] == season) & (historical_brain['TeamID'] == w_team)]
    l_stats = historical_brain[(historical_brain['Season'] == season) & (historical_brain['TeamID'] == l_team)]
    
    if w_stats.empty or l_stats.empty:
        continue
        
    w_adj_net = w_stats['AdjNetRating'].values[0]
    w_rank = w_stats['MedianRank'].values[0]
    l_adj_net = l_stats['AdjNetRating'].values[0]
    l_rank = l_stats['MedianRank'].values[0]
    
    # Data Augmentation for Symmetry
    if random.random() > 0.5:
        # Winner is Team A
        X_train_uni.append([w_adj_net - l_adj_net, l_rank - w_rank]) # Rank diff is flipped (lower is better)
        y_train_uni.append(1)
    else:
        # Loser is Team A
        X_train_uni.append([l_adj_net - w_adj_net, w_rank - l_rank])
        y_train_uni.append(0)

# Train the Universal Model
universal_model = LogisticRegression(C=1.0)
universal_model.fit(np.array(X_train_uni), np.array(y_train_uni))
print("Universal Model Trained on 2022-2024.")

# ==========================================
# 2. ISOLATE AND PREDICT 2025
# ==========================================
# Build the 2025 Brain
reg_2025_df = detailed_reg_df[detailed_reg_df['Season'] == 2025].copy()
ord_2025_df = ordinals_df[ordinals_df['Season'] == 2025].copy()
brain_2025 = build_efficiency_brain(reg_2025_df, ord_2025_df)

# Get actual 2025 Tournament matchups
tourney_2025 = tourney_df[tourney_df['Season'] == 2025].copy()
tourney_2025['Team1'] = tourney_2025[['WTeamID', 'LTeamID']].min(axis=1)
tourney_2025['Team2'] = tourney_2025[['WTeamID', 'LTeamID']].max(axis=1)
tourney_2025['ID'] = '2025_' + tourney_2025['Team1'].astype(str) + '_' + tourney_2025['Team2'].astype(str)

predictions = []

for _, row in tourney_2025.iterrows():
    team_a = row['Team1']
    team_b = row['Team2']
    
    a_stats = brain_2025[brain_2025['TeamID'] == team_a]
    b_stats = brain_2025[brain_2025['TeamID'] == team_b]
    
    # Safe fallback if a team somehow has no stats
    a_net = a_stats['AdjNetRating'].values[0] if not a_stats.empty else 0
    a_rank = a_stats['MedianRank'].values[0] if not a_stats.empty else 350
    b_net = b_stats['AdjNetRating'].values[0] if not b_stats.empty else 0
    b_rank = b_stats['MedianRank'].values[0] if not b_stats.empty else 350
    
    # Diff calculation exactly as in training
    net_diff = a_net - b_net
    rank_diff = b_rank - a_rank
    
    # Predict
    prob = universal_model.predict_proba([[net_diff, rank_diff]])[0][1]
    predictions.append({'ID': row['ID'], 'Pred': prob})

# ==========================================
# 3. EXPORT TO CSV
# ==========================================
universal_2025_df = pd.DataFrame(predictions)
universal_2025_df.to_csv('universal_2025_predictions.csv', index=False)
print("Saved 'universal_2025_predictions.csv' to disk. Ready for the Holy Grail notebook.")